# landcover_lca as geo_goblin uses it

geo_goblin (`impact_categories/climate_change.py`, class `ClimateChangeLandUse`, and
`impact_categories/eutrophication.py`, class `EutrophicationLandUse`) uses the **geo tree**,
`landcover_lca.geo_goblin.geo_models` and `landcover_lca.geo_goblin.geo_lca_emissions`, for farm or catchment areas.
This notebook makes the same calls with the same arguments on the test data in `geo_data/`.

The geo totals do **not** include off-site CO2 from horticultural peat: it is a national export figure and cannot be
assigned to a farm or catchment (it is in the national tree, used by goblin_lite).

**Conventions (as in geo_goblin):**
- Every function returns **kg** of gas (or of N, P, PO4e); geo_goblin multiplies by `kg_to_kt = 1e-6`.
- The baseline dataset is keyed `-calibration_year` (here `-2020`) and reported under scenario index `-1`.
  Scenarios are keyed `0, 1, ...` and reported in the target year.
- Baseline: current and past land use are both the baseline data. Scenario: current land use is the scenario,
  past land use is the baseline; the transition matrix of the same key holds the areas that change.
- Every function used here takes `(land_use_data, past_land_use_data, transition_matrix, ef_country)`.
  `total_ch4_emission_cropland` and `total_n2o_emission_cropland` took `(ef_country, transition_matrix,
  land_use_data, past_land_use_data)` before; the old order now raises TypeError. geo_goblin still calls them in the old
  order, and in its scenarios passes the baseline as current land use: update both when it moves to this version.

Run from `tests/` (the data paths are relative).

In [ ]:
import pandas as pd
import landcover_lca.geo_goblin.geo_lca_emissions as landuse_lca
import landcover_lca.geo_goblin.geo_models as landuse_models

ef_country = "ireland"
calibration_year = 2020
target_year = 2050
kg_to_kt = 1e-6
baseline_index = -1         # geo_goblin CommonParams.baseline_index
base = -calibration_year    # key of the baseline dataset

transition_data = pd.read_csv("./geo_data/transition_matrix.csv", index_col=0)
landuse_data = pd.read_csv("./geo_data/land_uses.csv", index_col=0)

transition_matrix = landuse_models.load_transition_matrix(transition_data, ef_country, calibration_year, target_year)
land_use_data = landuse_models.load_land_use_data(landuse_data, calibration_year)

scenarios = [int(k) for k in land_use_data.keys() if int(k) >= 0]
print("datasets:", list(land_use_data.keys()), "scenarios:", scenarios)

## Climate change: the functions geo_goblin calls
The three totals and one total per land use and gas.

In [ ]:
# One function per land use and gas, and the totals: the calls geo_goblin makes.
FUNCTIONS = {
    ("cropland", "CO2"): landuse_lca.total_co2_emission_cropland,
    ("cropland", "CH4"): landuse_lca.total_ch4_emission_cropland,
    ("cropland", "N2O"): landuse_lca.total_n2o_emission_cropland,
    ("grassland", "CO2"): landuse_lca.total_co2_emission_grassland,
    ("grassland", "CH4"): landuse_lca.total_ch4_emission_grassland,
    ("grassland", "N2O"): landuse_lca.total_n2o_emission_grassland,
    ("forest", "CO2"): landuse_lca.total_co2_emission_forest,
    ("forest", "CH4"): landuse_lca.total_ch4_emission_forest,
    ("forest", "N2O"): landuse_lca.total_n2o_emission_forest,
    ("wetland", "CO2"): landuse_lca.total_co2_emission_wetland,
    ("wetland", "CH4"): landuse_lca.total_ch4_emission_wetland,
    ("wetland", "N2O"): landuse_lca.total_n2o_emission_wetland,
    ("total", "CO2"): landuse_lca.total_co2_emission,
    ("total", "CH4"): landuse_lca.total_ch4_emission,
    ("total", "N2O"): landuse_lca.total_n2o_emission,
}
LAND_USES = ["cropland", "grassland", "forest", "wetland", "total"]


def landuse_emissions(key, scenario_index, year):
    """kt of each gas by land use for one dataset (key), as ClimateChangeLandUse builds it."""
    df = pd.DataFrame(
        columns=["CO2", "CH4", "N2O"],
        index=pd.MultiIndex.from_product([[scenario_index], LAND_USES, [year]], names=["scenario", "land_use", "year"]),
        dtype=float,
    )
    for (land_use, gas), function in FUNCTIONS.items():
        kg = function(land_use_data[key], land_use_data[base], transition_matrix[key], ef_country)
        df.loc[(scenario_index, land_use, year), gas] = kg * kg_to_kt
    return df

### Baseline (`climate_change_landuse_past`)

In [ ]:
past = landuse_emissions(base, baseline_index, calibration_year)
past

### Scenarios (`climate_change_landuse_future`)

In [ ]:
future = pd.concat([landuse_emissions(sc, sc, target_year) for sc in scenarios])
future

### CO2e
geo_goblin takes the GWP values from its `GoblinDataManager.get_AR_values()` (AR5: CH4 28, N2O 265).

In [ ]:
AR_values = {"CH4": 28, "N2O": 265}  # AR5, as in geo_goblin's GoblinDataManager (illustration)

emissions = pd.concat([past, future])
emissions["CO2e"] = emissions["CO2"] + emissions["CH4"] * AR_values["CH4"] + emissions["N2O"] * AR_values["N2O"]
emissions

### Drained organic grassland N2O
geo_goblin calls `drainage_n2O_organic_soils_in_grassland` in its agriculture emissions (`soil_histosol_N_direct`), as
the NID reports it in Agriculture 3.D. It is not part of `total_n2o_emission_grassland`, so it is not counted twice.

In [ ]:
# N2O from drained organic grassland soils, which geo_goblin reports in agriculture as "soil_histosol_N_direct".
histosol_n2o_kt = {
    key: landuse_lca.drainage_n2O_organic_soils_in_grassland(
        land_use_data[key], land_use_data[base], transition_matrix[key], ef_country
    ) * kg_to_kt
    for key in [base] + scenarios
}
histosol_n2o_kt

### Forest carbon from CBM
As goblin_lite, geo_goblin adds the CBM forest carbon (`"Total Ecosystem"`) to forest CO2 and to the total CO2. That
step is not shown here. Forest fire CO2 is therefore off by default (`include_fire_co2=False`; CBM's Wildfire
disturbance counts it); pass `include_fire_co2=True` to the CO2 totals for a run without CBM.

## Eutrophication (`EutrophicationLandUse`): forest exports to water
Forest only, in kg N, kg P and kg PO4e, converted to kt.

In [ ]:
EUTROPHICATION = {
    "N": landuse_lca.exports_to_water_N_forest,
    "P": landuse_lca.exports_to_water_P_forest,
    "total_PO4e": landuse_lca.exports_to_water_PO4e_forest,
}

rows = []
for key, scenario_index, year in [(base, baseline_index, calibration_year)] + [(sc, sc, target_year) for sc in scenarios]:
    row = {"scenario": scenario_index, "land_use": "forest", "year": year}
    for column, function in EUTROPHICATION.items():
        row[column] = function(land_use_data[key], land_use_data[base], transition_matrix[key], ef_country) * kg_to_kt
    rows.append(row)

eutrophication = pd.DataFrame(rows).set_index(["scenario", "land_use", "year"])
eutrophication